In [0]:
from pyspark.sql import functions as F

CATALOG = dbutils.widgets.get("catalog")
RAW_SCHEMA = dbutils.widgets.get("stream_schema")
TITLE = dbutils.widgets.get("title")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TITLE}"

### Checking quality rules

In [0]:
quality_results = (
    spark.table(SILVER_TABLE)
    .select(
        F.sum(
            F.when(
                F.col("show_id").isNull()
                | (F.length(F.trim(F.col("show_id"))) == 0),
                1
            ).otherwise(0)
        ).alias("invalid_show_id"),

        F.sum(
            F.when(
                F.col("title").isNull()
                | (F.length(F.trim(F.col("title"))) == 0),
                1
            ).otherwise(0)
        ).alias("invalid_title"),

        F.sum(
            F.when(
                F.col("type").isNull()
                | ~F.col("type").isin("Movie", "TV Show"),
                1
            ).otherwise(0)
        ).alias("invalid_type"),

        F.sum(
            F.when(
                F.col("release_year").isNull()
                | ~F.col("release_year").between(
                    1900,
                    F.year(F.current_date())
                ),
                1
            ).otherwise(0)
        ).alias("invalid_release_year")
    )
)

In [0]:
display(quality_results)

In [0]:
silver_after_rerun_df = spark.table(SILVER_TABLE)
after_rerun_count = silver_after_rerun_df.count()
print(f"After rerun: {after_rerun_count}")

In [0]:
duplicates_count = (
    silver_after_rerun_df
    .groupBy("show_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(duplicates_count)

In [0]:
detail_before = spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")

display(
    detail_before.select(
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:

spark.sql(f"""
    OPTIMIZE {SILVER_TABLE}
""")

In [0]:
detail_after = spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")

display(
    detail_after.select(
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
count_after_optimize = spark.table(SILVER_TABLE).count()

print(f"Before OPTIMIZE: {after_rerun_count}")
print(f"After OPTIMIZE:  {count_after_optimize}")

In [0]:
spark.sql(f"""
          VACUUM {SILVER_TABLE} RETAIN 168 HOURS
          """)

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
)